Walk through the process of reading a video and adding landmark overlays for each frame.

## Setup

In [ ]:
import sys
from pathlib import Path

import cv2
import mediapipe as mp
from IPython.display import Image, display

print("Python executable:", sys.executable)
print("OpenCV version:", cv2.__version__)
print("MediaPipe version:", mp.__version__)

In [ ]:
#Get file paths
# Notebook working directories can differ between editors.
cwd = Path.cwd()

if (cwd / "data" / "raw").is_dir():
    project_root = cwd
elif (cwd.parent / "data" / "raw").is_dir():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        f"Could not locate the project from {cwd}. "
        "Set project_root to your pushup-verifier directory."
    )

video_path = project_root / "data" / "raw" / "pilot_full_reps.MOV"
model_path = project_root / "models" / "pose_landmarker_full.task"

assert video_path.is_file(), f"Missing video: {video_path}"
assert model_path.is_file(), f"Missing model: {model_path}"

print("Project:", project_root)
print("Video:", video_path)
print("Model:", model_path)

## Read in and capture video

We will use cv2.VideoCapture to read video file.

Note: `cv2` is the python package for OpenCV.  To install it, need to run `pip install opencv-python`. 

In [ ]:
#No docstring!
cv2.VideoCapture?

In [ ]:
capture = cv2.VideoCapture(str(video_path))
type(capture)

In [ ]:
capture.isOpened()

In [ ]:
#First check is often to confirm it was opened successfully
if not capture.isOpened():
    raise RuntimeError(f"Failed to open video: {video_path}")
else:
    print(f"Successfully opened video: {video_path}")

In [ ]:
#Can get useful information about the video capture object
# print(f"Video width: {capture.get(cv2.CAP_PROP_WIDTH)}")
# print(f"Video height: {capture.get(cv2.CAP_PROP_HEIGHT)}")
print(f"Video FPS: {capture.get(cv2.CAP_PROP_FPS)}")
print(f"Video total frames: {capture.get(cv2.CAP_PROP_FRAME_COUNT)}")
print(f"Video duration (seconds): {capture.get(cv2.CAP_PROP_FRAME_COUNT) / capture.get(cv2.CAP_PROP_FPS)}")

In [ ]:
#Read a single frame
ret, frame = capture.read()

#ret is a boolean indicating whether the frame was read successfully
#frame is the actual image data of the frame
print(f"Frame read successfully: {ret}")
print(f"Frame shape: {frame.shape}")
print(f"Frame dtype: {frame.dtype}")

In [ ]:
#Each frame is a ndarray!
type(frame)

In [ ]:
#Complete cell to read and display video frame information
capture = cv2.VideoCapture(str(video_path))

try:
    if not capture.isOpened():
        raise RuntimeError(f"Could not open {video_path}")

    fps = capture.get(cv2.CAP_PROP_FPS)
    reported_frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

    success, frame_bgr = capture.read()

    if not success:
        raise RuntimeError("The video opened, but no frame could be read.")

finally:
    capture.release()

print("Reported FPS:", fps)
print("Reported frame count:", reported_frame_count)
print("Frame shape:", frame_bgr.shape)
print("Pixel data type:", frame_bgr.dtype)

In [ ]:
#The purpose of capture.release is to release the resources used by the video capture object.

One big quirk: OpenCV uses BGR color format by default, not RGB!

If you want to display the frame correctly in RGB, you need to convert it first.

In [ ]:
#One big quirk: OpenCV uses BGR color format by default, not RGB!
#If you want to display the frame correctly in RGB, you need to convert it first.
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

#Or could conver to grayscale
frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#This will squeeze out the 3rd dimension
frame_gray.shape

Another quirk: `cv.imshow()` method often crashes kernel when using it in a jupyter notebook, because the way it works tends to conflict with the interactive nature of notebooks.  Much better off using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
#Convert numpy array to matplotlib image
plt.imshow(frame_rgb)
plt.axis('off')
plt.show()

In [ ]:
#If you tried to display the BGR one...
plt.imshow(frame)
plt.axis('off')
plt.show()

In [ ]:
#And grayscale
plt.imshow(frame_gray, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
#Here's how to iterate through the frames of the video
capture = cv2.VideoCapture(str(video_path))
frame_count = 0
while True:
    ret, frame = capture.read()
    if not ret:
        break
    # Process the frame here
    if frame_count % 100 == 0:
        print(f"Processing frame {frame_count}")
    frame_count += 1
capture.release()
print(f"Total frames processed: {frame_count}")

In [ ]:
#What if you tried to read ANOTHER ONE
#It WON'T work because we already reached the end of the video
ret, frame = capture.read()
ret

In [ ]:
#Can we still get frame count after capture was released?
print(f"Video total frames: {capture.get(cv2.CAP_PROP_FRAME_COUNT)}")

So after running capture.release(), we can no longer get properties of the capture object, interseting!

In [ ]:
#Back to script to read in video
capture = cv2.VideoCapture(str(video_path))

try:
    if not capture.isOpened():
        raise RuntimeError(f"Could not open {video_path}")

    fps = capture.get(cv2.CAP_PROP_FPS)
    reported_frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

    success, frame_bgr = capture.read()

    if not success:
        raise RuntimeError("The video opened, but no frame could be read.")

finally:
    capture.release()

print("Reported FPS:", fps)
print("Reported frame count:", reported_frame_count)
print("Frame shape:", frame_bgr.shape)
print("Pixel data type:", frame_bgr.dtype)

In [ ]:
#Display image.  
#Before we converted BGR to RGB and used plt.imshow()
#Here's another approach (recommended by ChatGPT) to conver tto png
def show_bgr(image_bgr, display_width=700):
    success, encoded_png = cv2.imencode(".png", image_bgr)

    if not success:
        raise RuntimeError("Could not encode the image for display.")

    display(Image(data=encoded_png.tobytes(), width=display_width))


show_bgr(frame_bgr)

## Feed into MediaPipe to get landmark locations

In [ ]:
#Convert BGR to RGB
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

#still have ndarray
print("Frame shape:", frame_rgb.shape)
print("Pixel data type:", frame_rgb.dtype)

In [ ]:
#convert to MediaPipe image
mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=frame_rgb,
)

print("MediaPipe image size:", mp_image.width, "×", mp_image.height)

In [ ]:
type(mp_image)

In [ ]:
#Now load the MediaPipe Pose model
#this points to the Full Landmark Pose model we already downloaded
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=str(model_path)
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_poses=1,
)

#Create instance of the PoseLandmarker, and use it to detect poses in the input image
with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
    result = landmarker.detect(mp_image)

print("Number of detected poses:", len(result.pose_landmarks))

The `with` context ensures that the bulky resources of the Landmarker model are released once it is done generatingn the results.

The `num_poses = 1` means we are only looking for 1 person.

In [ ]:
#result is now a PoseLandmarkerResult object containing the detected poses and their landmarks
type(result)

In [ ]:
#Get list of all 33 landmarks for the first detected pose
#Technically it is a list of lists, with each inner list being a pose
#Here we have one pose, so one inner list containing 33 landmarks
landmark_list = result.pose_landmarks[0]
print("Number of landmarks:", len(landmark_list))
landmark_list[:5]

In [ ]:
#Check out first pose
pose = landmark_list[0]
pose

For each pose:
- x and y are normalized from 0 to 1, so normalized units for the boundaries of the image (x = 1 is right, y = 1 is top)
- Technically x and y can be outside of the 0 to 1 range if it predicts a landmark outside of the bounds of the image
- z gives the depth, where 0 represents the center of the hip, and negative means closer to the camera.  The magnitude is roughly the same scale as the x variable.  So here we have z = -0.65, meaning that landmark is closer to the camera than the hip midpoint.  The first 11 landmarks are all on the face, so here the face is closer to camera than the hip midpoint, so negative number makes sense.
- visible is whether that landmark is currently visible in the frame rather than being hidden
- presence is whether thta landmark is in the bounds of the image scene

Let's see what landarks are and are not "visible", and see if that lines up with the 33 landmarks noted at https://developers.google.com/edge/mediapipe/solutions/vision/pose_landmarker:

```text
0 - nose
1 - left eye (inner)
2 - left eye
3 - left eye (outer)
4 - right eye (inner)
5 - right eye
6 - right eye (outer)
7 - left ear
8 - right ear
9 - mouth (left)
10 - mouth (right)
11 - left shoulder
12 - right shoulder
13 - left elbow
14 - right elbow
15 - left wrist
16 - right wrist
17 - left pinky
18 - right pinky
19 - left index
20 - right index
21 - left thumb
22 - right thumb
23 - left hip
24 - right hip
25 - left knee
26 - right knee
27 - left ankle
28 - right ankle
29 - left heel
30 - right heel
31 - left foot index
32 - right foot index
```

In [ ]:
#Most visible
import numpy as np
landmark_visbilities = [x.visibility for x in landmark_list]
most_visible = np.argmax(landmark_visbilities)

#Least visible
least_visible = np.argmin(landmark_visbilities)

print('Most visible landmark index:', most_visible)
print('Least visible landmark index:', least_visible)
print('Most visible landmark:', landmark_list[most_visible])
print('Least visible landmark:', landmark_list[least_visible])


Most visible is index 6 which is right eye.  Least visible is 25, which is left knee.  Checks out!

In [ ]:
#Back to finding the right-side landmarks
if not result.pose_landmarks:
    raise RuntimeError(
        "No pose detected in this frame. We'll need to try another frame."
    )

landmarks = result.pose_landmarks[0]

right_side = {
    "shoulder": 12,
    "elbow": 14,
    "wrist": 16,
    "hip": 24,
    "ankle": 28,
}

for name, index in right_side.items():
    landmark = landmarks[index]

    print(
        f"{name:8s} | "
        f"x={landmark.x:.3f}, "
        f"y={landmark.y:.3f}, "
        f"visibility={landmark.visibility:.3f}, "
        f"presence={landmark.presence:.3f}"
    )

In [ ]:
#Convert normalized coordinates to x-y pixel coordinates
height, width = frame_bgr.shape[:2]

points = {}

for name, index in right_side.items():
    landmark = landmarks[index]

    points[name] = (
        landmark.x * width,
        landmark.y * height,
    )

points

In [ ]:
#Keep these float pixel values for now since they will hepl with more accurate calculations
#Can also build this handy drawing function to convert to int's for drawing
draw_points = {
    name: (round(x), round(y))
    for name, (x, y) in points.items()
}

print("For calculations:", points["elbow"])
print("For drawing:", draw_points["elbow"])

In [ ]:
#Prepare for drawing the pose landmarks on the frame
# Preserve the original frame.
annotated_frame = frame_bgr.copy()

# Round only the coordinates used for drawing.
draw_points = {
    name: (round(x), round(y))
    for name, (x, y) in points.items()
}

# OpenCV uses BGR color order.
green = (0, 255, 0)
yellow = (0, 255, 255)
white = (255, 255, 255)

In [ ]:
connections = [
    ("shoulder", "elbow"),
    ("elbow", "wrist"),
    ("shoulder", "hip"),
    ("hip", "ankle"),
]

# Draw the connecting lines first.
for start_name, end_name in connections:
    cv2.line(
        annotated_frame,
        draw_points[start_name],
        draw_points[end_name],
        color=green,
        thickness=3,
        lineType=cv2.LINE_AA,
    )

# Draw a circle and label at each landmark.
for name, (x, y) in draw_points.items():
    cv2.circle(
        annotated_frame,
        center=(x, y),
        radius=6,
        color=yellow,
        thickness=-1,
        lineType=cv2.LINE_AA,
    )

    cv2.putText(
        annotated_frame,
        text=name,
        org=(x + 10, y - 10),
        fontFace=cv2.FONT_HERSHEY_SIMPLEX,
        fontScale=0.6,
        color=white,
        thickness=2,
        lineType=cv2.LINE_AA,
    )

show_bgr(annotated_frame)

In [ ]:
#Calculate angle
import numpy as np


def angle_at_joint(a, b, c):
    """Calculate angle ABC in degrees, with B as the joint."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)

    if not np.isfinite([a, b, c]).all():
        raise ValueError("Coordinates must be finite.")

    # Both vectors originate at the joint.
    u = a - b
    v = c - b

    length_u = np.linalg.norm(u)
    length_v = np.linalg.norm(v)

    if length_u < 1e-8 or length_v < 1e-8:
        raise ValueError("Cannot calculate an angle from coincident points.")

    cosine = np.dot(u, v) / (length_u * length_v)

    # Protect arccos against tiny floating-point errors beyond [-1, 1].
    cosine = np.clip(cosine, -1.0, 1.0)

    return float(np.degrees(np.arccos(cosine)))

In [ ]:
#Check
print("Right angle:", angle_at_joint((1, 0), (0, 0), (0, 1)))
print("Straight line:", angle_at_joint((-1, 0), (0, 0), (1, 0)))

In [ ]:
# Use the original float coordinates for geometry.
elbow_angle = angle_at_joint(
    points["shoulder"],
    points["elbow"],
    points["wrist"],
)

print(f"Right elbow angle: {elbow_angle:.2f} degrees")
print(f"Meets the single-frame depth threshold: {elbow_angle <= 90.0}")

# Add a separate header so the measurement doesn't cover your joints.
angle_frame = cv2.copyMakeBorder(
    annotated_frame,
    top=100,
    bottom=0,
    left=0,
    right=0,
    borderType=cv2.BORDER_CONSTANT,
    value=(25, 25, 25),
)

cv2.putText(
    angle_frame,
    f"Right elbow: {elbow_angle:.1f} deg",
    (15, 38),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.7,
    white,
    2,
    cv2.LINE_AA,
)

cv2.putText(
    angle_frame,
    "Depth threshold: <= 90 deg",
    (15, 78),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    yellow,
    2,
    cv2.LINE_AA,
)

show_bgr(angle_frame)
